# Phase 9: value bets + backtest

Rules (set in `nba_props/config.py`): bet a side only if **EV = P(side) x decimal odds - 1 > 5%**, flat 1-unit stakes, **at most one bet per player per game**. The pipeline places bets at the DraftKings price it saw, and the latest pre-game line becomes the closing line for CLV.

Report: bets, win rate, profit, ROI, ROI by month, max drawdown, bootstrap CI on ROI, CLV. Gate 3 needs positive CLV and no clearly negative ROI after 300+ bets.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nba_props.config import EV_THRESHOLD, LIVE_DIR
from nba_props.evaluate import bet_summary, bootstrap_roi, by_period, max_drawdown
from nba_props.odds import american_to_decimal

pd.set_option("display.max_columns", 50)
path = LIVE_DIR / "paper_bets.parquet"
bets = pd.read_parquet(path) if path.exists() else pd.DataFrame(columns=["status", "profit", "clv", "ev", "start_time"])
settled = bets[bets["status"].isin(["won", "lost"])].sort_values("start_time")
print(len(bets), "paper bets;", bets["status"].value_counts().to_dict())

## Step 1: headline results

In [ ]:
pd.Series(bet_summary(bets))

## Step 2: by month and the profit curve

In [ ]:
display(by_period(bets).round(3))
if len(settled):
    plt.plot(settled["profit"].cumsum().to_numpy())
    plt.axhline(0, ls="--", c="grey"); plt.xlabel("bet #"); plt.ylabel("units"); plt.title("Cumulative paper profit")

## Step 3: is the edge real? CLV first, ROI second
ROI on a few hundred bets is mostly noise (the bootstrap interval will be wide). CLV settles down much faster: if our prices beat the close on average, the model is finding something the market later agrees with.

In [ ]:
if len(settled) >= 2:
    lo, hi = bootstrap_roi(settled["profit"])
    print(f"ROI {settled['profit'].mean():+.3f}, 95% CI {lo:+.3f} to {hi:+.3f}, max drawdown {max_drawdown(settled['profit']):.1f}u")
    clv = settled["clv"].dropna()
    if len(clv):
        print(f"CLV mean {clv.mean():+.4f}, share positive {(clv > 0).mean():.2f}, n={len(clv)}")
else:
    print("no settled bets yet")

## Step 4: who are we betting on?
Model 1 warning sign: the biggest-EV bets clustered on one kind of team (promoted sides), and those were the model being wrong. Check by side, by opening game, by EV size and by line size.

In [ ]:
if len(settled):
    s = settled.copy()
    s["ev_bucket"] = pd.cut(s["ev"], [0, 0.1, 0.2, 0.4, 10])
    s["line_bucket"] = pd.cut(s["line"], [0, 10, 20, 30, 100])
    for col in ["side", "season_start", "ev_bucket", "line_bucket"]:
        if col in s:
            display(s.groupby(col, observed=True).agg(bets=("profit", "size"), roi=("profit", "mean"), clv=("clv", "mean")).round(3))
else:
    print("no settled bets yet")

## Step 5: threshold sweep on every predicted prop
The paper bets use one threshold. Here every finished prop is re-scored at several thresholds using the first price we saw, which shows whether 5% is too low (the vig eats it) or too high (too few bets). It uses only pre-game information, so it's a fair backtest of the rule.

In [ ]:
ppath = LIVE_DIR / "predictions.parquet"
preds = pd.read_parquet(ppath) if ppath.exists() else pd.DataFrame()
if len(preds) and "played" in preds and (preds["played"] == True).sum():  # noqa: E712
    d = preds[(preds["played"] == True) & preds["first_p_over"].notna()].copy()  # noqa: E712
    dec_o, dec_u = american_to_decimal(d["first_over_odds"]), american_to_decimal(d["first_under_odds"])
    ev_o, ev_u = d["first_p_over"] * dec_o - 1, (1 - d["first_p_over"]) * dec_u - 1
    over_side = (ev_o >= ev_u).to_numpy()
    best_ev = np.where(over_side, ev_o, ev_u)
    won = np.where(over_side, d["actual_points"] > d["first_line"], d["actual_points"] < d["first_line"])
    profit = np.where(won, np.where(over_side, dec_o, dec_u) - 1, -1.0)
    rows = []
    for t in [0, 0.02, 0.05, 0.08, 0.12, 0.2]:
        m = best_ev > t
        rows.append({"threshold": t, "bets": int(m.sum()), "roi": profit[m].mean() if m.any() else np.nan})
    display(pd.DataFrame(rows).round(3))
    print("current threshold:", EV_THRESHOLD)
else:
    print("no finished props yet")

## Step 6: Gate 3 verdict
Log to `nba/docs/experiments.md`. Pass = positive average CLV and no clearly negative ROI (CI upper bound above 0) after 300+ settled bets.

In [ ]:
n = len(settled)
if n < 300:
    print(f"{n} settled bets: keep paper trading (Gate 3 needs 300+)")
else:
    s = bet_summary(bets)
    ok = (s["avg_clv"] or 0) > 0 and s["roi_ci95"][1] > 0
    print("GATE 3", "PASSES" if ok else "FAILS", s)